# BEV Electrification Scenario – Decent Mobility CO₂

This notebook replicates the car decent-mobility calculation with a BEV emission factor (70 g CO₂/km instead of 162 g CO₂/km for ICE), reads the ICE baseline results, and produces:
- Per-city boxplot comparing ICE vs BEV decent mobility CO₂
- Cross-city bar chart showing % of users under the 2030 carbon budget (7 kg/week)

## Imports and constants

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from h3 import h3

BUDGET_2030  = 7000    # g CO2/week
BUDGET_2050  = 3000    # g CO2/week
ICE_FACTOR   = 162     # g CO2/km — original
BEV_FACTOR   = 70      # g CO2/km — electric
SCALE        = BEV_FACTOR / ICE_FACTOR   # ≈ 0.432


## Helper: rescale CO₂ matrix from ICE to BEV

In [ ]:
def rescale_car_co2(df_car_co2, scale=SCALE):
    """
    The original car_co2 column = distance * 162.
    BEV equivalent = distance * 70 = original * (70/162).
    Applies to outbound, inbound and total columns.
    """
    df = df_car_co2.copy()
    for col in ["car_co2_outbound", "car_co2_inbound", "car_co2_total"]:
        if col in df.columns:
            df[col] = df[col] * scale
    return df


## Helper: compute BEV decent mobility for one city

In [ ]:
def run_bev(city, car_co2_path, car_typ_cat_path, users_path,
            jobs_path, pois_path, postal_path,
            postal_nimi_col="Nimi", postal_posnro_col="Posnro"):
    """
    Runs the full car decent-mobility pipeline with BEV-scaled CO2.
    Returns weekly_routine DataFrame with column 'decent_mobility_co2'.
    """
    print(f"\n{'='*50}\n{city}\n{'='*50}")

    # ── USERS ──────────────────────────────────────────────────────────────
    users = pd.read_parquet(users_path)
    users = users[users["home_gid9"].notna()]

    # ── JOBS + POIS ────────────────────────────────────────────────────────
    jobs = gpd.read_parquet(jobs_path).reset_index()
    pois = pd.read_parquet(pois_path)

    users = users.merge(
        jobs[["ID", "weighted_tyo"]],
        left_on="stay_gid9", right_on="ID", how="left"
    ).drop(columns="ID")

    pois_grouped = (
        pois.groupby(["h3_id", "category"])["count"]
        .sum().unstack(fill_value=0).reset_index()
    )
    activity_cols = ["Social, Cultural", "Shopping, Errands", "Recreational, Outdoors"]
    for col in activity_cols:
        if col not in pois_grouped.columns:
            pois_grouped[col] = 0

    users = users.merge(
        pois_grouped[["h3_id"] + activity_cols],
        left_on="stay_gid9", right_on="h3_id", how="left"
    ).drop(columns="h3_id")

    # ── POSTAL ────────────────────────────────────────────────────────────
    postal = gpd.read_file(postal_path)
    users["geometry"] = users["home_gid9"].apply(
        lambda h: Polygon(h3.h3_to_geo_boundary(h, geo_json=True))
    )
    gdf = gpd.GeoDataFrame(users, geometry="geometry", crs="EPSG:4326").to_crs(postal.crs)
    ints = gpd.overlay(gdf, postal, how="intersection")
    ints["intersect_area"] = ints.geometry.area
    idx = ints.groupby("home_gid9")["intersect_area"].idxmax()
    largest = ints.loc[idx]
    gdf = gdf.merge(
        largest[["home_gid9", postal_posnro_col, postal_nimi_col, "geometry"]],
        on="home_gid9", how="left"
    ).rename(columns={postal_nimi_col: "Nimi", postal_posnro_col: "Posnro"})

    # ── CAR CO2 — load and rescale to BEV ──────────────────────────────────
    cols_needed = ["Origin_Hexagon_ID", "Destination_Hexagon_ID", "car_co2"]
    table = pq.read_table(car_co2_path, columns=cols_needed, use_threads=True)
    df_car = table.to_pandas(types_mapper=pd.ArrowDtype)
    df_car = df_car.rename(columns={
        "Origin_Hexagon_ID": "from_id",
        "Destination_Hexagon_ID": "to_id",
        "car_co2": "car_co2_outbound"
    })
    # Symmetrise
    inbound = df_car.rename(columns={"from_id":"to_id","to_id":"from_id","car_co2_outbound":"car_co2_inbound"})
    df_sym = df_car.merge(inbound, on=["from_id","to_id"], how="left")
    df_sym["car_co2_inbound"] = df_sym["car_co2_inbound"].fillna(df_sym["car_co2_outbound"])
    df_sym["car_co2_total"]   = df_sym["car_co2_outbound"] + df_sym["car_co2_inbound"]
    df_sym = df_sym.rename(columns={"from_id_x":"from_id","to_id_x":"to_id"})
    if "from_id_x" in df_sym.columns:
        df_sym = df_sym.rename(columns={"from_id_x":"from_id","to_id_x":"to_id"})
    df_car_co2 = df_sym[["from_id","to_id","car_co2_outbound","car_co2_inbound","car_co2_total"]].copy()

    # Rescale ICE → BEV
    df_car_co2 = rescale_car_co2(df_car_co2)

    # ── TOUR PIPELINE ──────────────────────────────────────────────────────
    df_base = users[(users["is_home"]==0) & (users["is_work"]==0)].copy()
    df_long = df_base.melt(
        id_vars=["user_id","stay_gid9","home_gid9","work_gid9","frequency_period"],
        value_vars=activity_cols, var_name="activity_type", value_name="poi_count"
    )
    df_long = df_long[df_long["poi_count"] > 0].copy()
    df_long = df_long.rename(columns={"stay_gid9":"activity_gid9"})
    df_long["tour_id"] = df_long.groupby(["user_id","activity_type"]).cumcount()

    def build_legs(row):
        return [
            (row.user_id, row.home_gid9,     row.work_gid9,     row.activity_type, row.frequency_period, row.poi_count),
            (row.user_id, row.work_gid9,     row.activity_gid9, row.activity_type, row.frequency_period, row.poi_count),
            (row.user_id, row.activity_gid9, row.home_gid9,     row.activity_type, row.frequency_period, row.poi_count),
        ]
    legs = df_long.apply(build_legs, axis=1)
    legs_df = pd.DataFrame(
        [leg for tour in legs for leg in tour],
        columns=["user_id","from_id","to_id","tour_type","frequency","poi_count"]
    )
    legs_df = legs_df.merge(df_car_co2, on=["from_id","to_id"], how="left")
    legs_df["tour_id"] = legs_df.index // 3

    tour_validity = legs_df.groupby("tour_id").agg(n_legs=("car_co2_outbound","count")).reset_index()
    valid_tours   = tour_validity[tour_validity["n_legs"]==3]
    legs_df = legs_df.merge(valid_tours[["tour_id"]], on="tour_id", how="inner")

    tour_df = (
        legs_df.groupby(["user_id","tour_id","tour_type","frequency","poi_count"])["car_co2_outbound"]
        .sum().reset_index()
        .rename(columns={"car_co2_outbound":"tour_co2"})
    )
    tour_df["exposure"] = tour_df["frequency"] * tour_df["poi_count"]

    tour_counts = tour_df.groupby(["user_id","tour_type"]).size().reset_index(name="n_tours")
    valid_pairs = tour_counts[tour_counts["n_tours"]>=3]
    tour_df = tour_df.merge(valid_pairs[["user_id","tour_type"]], on=["user_id","tour_type"], how="inner")

    tour_df = tour_df.sort_values(["user_id","tour_type","tour_co2"])
    tour_df["cum_exposure"]       = tour_df.groupby(["user_id","tour_type"])["exposure"].cumsum()
    tour_df["total_exposure"]     = tour_df.groupby(["user_id","tour_type"])["exposure"].transform("sum")
    tour_df["cum_exposure_share"] = tour_df["cum_exposure"] / tour_df["total_exposure"]
    tour_df = tour_df.merge(
        gdf[["user_id","home_gid9","Posnro","Nimi"]].drop_duplicates(),
        on="user_id", how="left"
    )

    df_typical = (
        tour_df.sort_values(["user_id","tour_type","cum_exposure_share"])
        .loc[tour_df["cum_exposure_share"]>=0.5]
        .groupby(["user_id","tour_type","Nimi","Posnro"], as_index=False)
        .first()
        .rename(columns={"tour_co2":"typical_trip_co2"})
    )

    # ── SINGLE TRIP (BEV-scaled) ───────────────────────────────────────────
    df_typical_single = pd.read_parquet(car_typ_cat_path)
    # Rescale single trip co2 to BEV
    df_typical_single["typical_trip_co2"] = df_typical_single["typical_trip_co2"] * SCALE

    df_final = df_typical.merge(
        df_typical_single,
        left_on=["user_id","tour_type","Nimi","Posnro"],
        right_on=["user_id","poi_type","Nimi","Posnro"],
        how="left", suffixes=("","_single")
    ).drop(columns=["poi_type"], errors="ignore")

    df_jobs = df_typical_single[df_typical_single["poi_type"]=="jobs"].copy()
    df_jobs = df_jobs.rename(columns={"typical_trip_co2":"co2_jobs","poi_type":"tour_type"})
    df_jobs["tour_type"] = "jobs"
    df_jobs["tour_id"] = df_jobs["frequency"] = df_jobs["poi_count"] = 1
    df_jobs["exposure"] = df_jobs["cum_exposure"] = df_jobs["total_exposure"] = df_jobs["cum_exposure_share"] = 1
    df_jobs = df_jobs.rename(columns={"co2_jobs":"typical_trip_co2"})
    df_jobs["typical_trip_co2_single"] = df_jobs["typical_trip_co2"]
    for col in df_final.columns:
        if col not in df_jobs.columns:
            df_jobs[col] = 1
    df_jobs = df_jobs[df_final.columns]
    df_final = pd.concat([df_final, df_jobs], ignore_index=True)

    required = {"jobs","Social, Cultural","Shopping, Errands","Recreational, Outdoors"}
    valid_users = df_final.groupby("user_id")["tour_type"].apply(lambda x: required.issubset(set(x)))
    valid_users = valid_users[valid_users].index
    df = df_final[df_final["user_id"].isin(valid_users)].copy()

    tour   = df.pivot(index="user_id", columns="tour_type", values="typical_trip_co2")
    single = df.pivot(index="user_id", columns="tour_type", values="typical_trip_co2_single")

    weekly = pd.DataFrame(index=tour.index)
    weekly["social_component"]    = 2 * tour["Social, Cultural"]
    weekly["recreation_component"]= 2 * tour["Recreational, Outdoors"]
    weekly["shopping_component"]  = single["Shopping, Errands"]
    weekly["decent_mobility_co2"] = weekly[["social_component","recreation_component","shopping_component"]].sum(axis=1)
    weekly = weekly.reset_index()

    pct_2030 = (weekly["decent_mobility_co2"] <= BUDGET_2030).mean()*100
    pct_2050 = (weekly["decent_mobility_co2"] <= BUDGET_2050).mean()*100
    print(f"  Users: {len(weekly):,}  |  <2030 budget: {pct_2030:.1f}%  |  <2050 budget: {pct_2050:.1f}%")

    return weekly


## City configurations

In [ ]:
CITIES = {
    "Helsinki": dict(
        car_co2_path    = "scratch/car_co2_6000.parquet",
        car_typ_cat_path= "scratch/car_typ_cat.parquet",
        users_path      = "scratch/data/users_and_stays_3months.parquet",
        jobs_path       = "data/job_distribution_from_census.parquet",
        pois_path       = "data/pois_per_hex_new_class.parquet",
        postal_path     = "./data/postal_code/Postinumeroalueet_2024.shp",
    ),
    "Tampere": dict(
        car_co2_path    = "scratch/car_co2_6000_tampere.parquet",
        car_typ_cat_path= "scratch/car_typ_cat_tampere.parquet",
        users_path      = "scratch/data/users_and_stays_3months_tampere.parquet",
        jobs_path       = "data/job_distribution_from_census_tampere.parquet",
        pois_path       = "data/pois_per_hex_new_class_tampere.parquet",
        postal_path     = "./data/postal_code_finland/pno_tilasto_2024.shp",
        postal_nimi_col = "nimi", postal_posnro_col = "postinumer",
    ),
    "Turku": dict(
        car_co2_path    = "scratch/car_co2_6000_turku.parquet",
        car_typ_cat_path= "scratch/car_typ_cat_turku.parquet",
        users_path      = "scratch/data/users_and_stays_3months_turku.parquet",
        jobs_path       = "data/job_distribution_from_census_turku.parquet",
        pois_path       = "data/pois_per_hex_new_class_turku.parquet",
        postal_path     = "./data/postal_code_finland/pno_tilasto_2024.shp",
        postal_nimi_col = "nimi", postal_posnro_col = "postinumer",
    ),
    "Oulu": dict(
        car_co2_path    = "scratch/car_co2_6000_oulu.parquet",
        car_typ_cat_path= "scratch/car_typ_cat_oulu.parquet",
        users_path      = "scratch/data/users_and_stays_3months_oulu.parquet",
        jobs_path       = "data/job_distribution_from_census_oulu.parquet",
        pois_path       = "data/pois_per_hex_new_class_oulu.parquet",
        postal_path     = "./data/postal_code_finland/pno_tilasto_2024.shp",
        postal_nimi_col = "nimi", postal_posnro_col = "postinumer",
    ),
}


In [ ]:
bev_results = {}
for city, cfg in CITIES.items():
    bev_results[city] = run_bev(city, **cfg)


In [ ]:
# ── Helsinki ──────────────────────────────────────────────────────────────────
ice_helsinki = pd.read_parquet("./output/tour_car_helsinki_decent_per_user.parquet")
bev_helsinki = bev_results["Helsinki"]

fig, ax = plt.subplots(figsize=(9, 4))

data = [
    ice_helsinki["decent_mobility_co2"].dropna(),
    bev_helsinki["decent_mobility_co2"].dropna(),
]
bp = ax.boxplot(
    data, vert=False, widths=0.5, patch_artist=True,
    boxprops=dict(edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black", linewidth=2),
    flierprops=dict(marker=".", markersize=3, alpha=0.35)
)
bp["boxes"][0].set_facecolor("#b2182b")   # ICE red
bp["boxes"][1].set_facecolor("#2166ac")   # BEV blue

ax.axvline(BUDGET_2030, color="tab:orange", linestyle="--", linewidth=1.8, label="2030 budget (7 kg/week)")

pct_ice_2030 = (ice_helsinki["decent_mobility_co2"] <= BUDGET_2030).mean()*100
pct_bev_2030 = (bev_helsinki["decent_mobility_co2"] <= BUDGET_2030).mean()*100

ax.set_yticks([1, 2])
ax.set_yticklabels(["ICE", "BEV"], fontsize=12)
ax.set_xlabel("Weekly decent mobility CO₂ (g)", fontsize=11)
ax.set_title(f"Helsinki — ICE vs BEV decent mobility CO₂", fontsize=13)
ax.legend(frameon=False, fontsize=10)
ax.grid(axis="x", linestyle=":", alpha=0.4)
for s in ["top","right","left"]:
    ax.spines[s].set_visible(False)

ax.text(1.01, 0.75, f"ICE: {pct_ice_2030:.1f}% <2030",
        transform=ax.transAxes, fontsize=9, va="center", color="#b2182b")
ax.text(1.01, 0.25, f"BEV: {pct_bev_2030:.1f}% <2030",
        transform=ax.transAxes, fontsize=9, va="center", color="#2166ac")

plt.tight_layout()
plt.savefig("./output/bev_vs_ice_helsinki.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Tampere ──────────────────────────────────────────────────────────────────
ice_tampere = pd.read_parquet("./output/tour_car_tampere_decent_per_user.parquet")
bev_tampere = bev_results["Tampere"]

fig, ax = plt.subplots(figsize=(9, 4))

data = [
    ice_tampere["decent_mobility_co2"].dropna(),
    bev_tampere["decent_mobility_co2"].dropna(),
]
bp = ax.boxplot(
    data, vert=False, widths=0.5, patch_artist=True,
    boxprops=dict(edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black", linewidth=2),
    flierprops=dict(marker=".", markersize=3, alpha=0.35)
)
bp["boxes"][0].set_facecolor("#b2182b")   # ICE red
bp["boxes"][1].set_facecolor("#2166ac")   # BEV blue

ax.axvline(BUDGET_2030, color="tab:orange", linestyle="--", linewidth=1.8, label="2030 budget (7 kg/week)")

pct_ice_2030 = (ice_tampere["decent_mobility_co2"] <= BUDGET_2030).mean()*100
pct_bev_2030 = (bev_tampere["decent_mobility_co2"] <= BUDGET_2030).mean()*100

ax.set_yticks([1, 2])
ax.set_yticklabels(["ICE", "BEV"], fontsize=12)
ax.set_xlabel("Weekly decent mobility CO₂ (g)", fontsize=11)
ax.set_title(f"Tampere — ICE vs BEV decent mobility CO₂", fontsize=13)
ax.legend(frameon=False, fontsize=10)
ax.grid(axis="x", linestyle=":", alpha=0.4)
for s in ["top","right","left"]:
    ax.spines[s].set_visible(False)

ax.text(1.01, 0.75, f"ICE: {pct_ice_2030:.1f}% <2030",
        transform=ax.transAxes, fontsize=9, va="center", color="#b2182b")
ax.text(1.01, 0.25, f"BEV: {pct_bev_2030:.1f}% <2030",
        transform=ax.transAxes, fontsize=9, va="center", color="#2166ac")

plt.tight_layout()
plt.savefig("./output/bev_vs_ice_tampere.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Turku ──────────────────────────────────────────────────────────────────
ice_turku = pd.read_parquet("./output/tour_car_turku_decent_per_user.parquet")
bev_turku = bev_results["Turku"]

fig, ax = plt.subplots(figsize=(9, 4))

data = [
    ice_turku["decent_mobility_co2"].dropna(),
    bev_turku["decent_mobility_co2"].dropna(),
]
bp = ax.boxplot(
    data, vert=False, widths=0.5, patch_artist=True,
    boxprops=dict(edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black", linewidth=2),
    flierprops=dict(marker=".", markersize=3, alpha=0.35)
)
bp["boxes"][0].set_facecolor("#b2182b")   # ICE red
bp["boxes"][1].set_facecolor("#2166ac")   # BEV blue

ax.axvline(BUDGET_2030, color="tab:orange", linestyle="--", linewidth=1.8, label="2030 budget (7 kg/week)")

pct_ice_2030 = (ice_turku["decent_mobility_co2"] <= BUDGET_2030).mean()*100
pct_bev_2030 = (bev_turku["decent_mobility_co2"] <= BUDGET_2030).mean()*100

ax.set_yticks([1, 2])
ax.set_yticklabels(["ICE", "BEV"], fontsize=12)
ax.set_xlabel("Weekly decent mobility CO₂ (g)", fontsize=11)
ax.set_title(f"Turku — ICE vs BEV decent mobility CO₂", fontsize=13)
ax.legend(frameon=False, fontsize=10)
ax.grid(axis="x", linestyle=":", alpha=0.4)
for s in ["top","right","left"]:
    ax.spines[s].set_visible(False)

ax.text(1.01, 0.75, f"ICE: {pct_ice_2030:.1f}% <2030",
        transform=ax.transAxes, fontsize=9, va="center", color="#b2182b")
ax.text(1.01, 0.25, f"BEV: {pct_bev_2030:.1f}% <2030",
        transform=ax.transAxes, fontsize=9, va="center", color="#2166ac")

plt.tight_layout()
plt.savefig("./output/bev_vs_ice_turku.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Oulu ──────────────────────────────────────────────────────────────────
ice_oulu = pd.read_parquet("./output/tour_car_oulu_decent_per_user.parquet")
bev_oulu = bev_results["Oulu"]

fig, ax = plt.subplots(figsize=(9, 4))

data = [
    ice_oulu["decent_mobility_co2"].dropna(),
    bev_oulu["decent_mobility_co2"].dropna(),
]
bp = ax.boxplot(
    data, vert=False, widths=0.5, patch_artist=True,
    boxprops=dict(edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black", linewidth=2),
    flierprops=dict(marker=".", markersize=3, alpha=0.35)
)
bp["boxes"][0].set_facecolor("#b2182b")   # ICE red
bp["boxes"][1].set_facecolor("#2166ac")   # BEV blue

ax.axvline(BUDGET_2030, color="tab:orange", linestyle="--", linewidth=1.8, label="2030 budget (7 kg/week)")

pct_ice_2030 = (ice_oulu["decent_mobility_co2"] <= BUDGET_2030).mean()*100
pct_bev_2030 = (bev_oulu["decent_mobility_co2"] <= BUDGET_2030).mean()*100

ax.set_yticks([1, 2])
ax.set_yticklabels(["ICE", "BEV"], fontsize=12)
ax.set_xlabel("Weekly decent mobility CO₂ (g)", fontsize=11)
ax.set_title(f"Oulu — ICE vs BEV decent mobility CO₂", fontsize=13)
ax.legend(frameon=False, fontsize=10)
ax.grid(axis="x", linestyle=":", alpha=0.4)
for s in ["top","right","left"]:
    ax.spines[s].set_visible(False)

ax.text(1.01, 0.75, f"ICE: {pct_ice_2030:.1f}% <2030",
        transform=ax.transAxes, fontsize=9, va="center", color="#b2182b")
ax.text(1.01, 0.25, f"BEV: {pct_bev_2030:.1f}% <2030",
        transform=ax.transAxes, fontsize=9, va="center", color="#2166ac")

plt.tight_layout()
plt.savefig("./output/bev_vs_ice_oulu.png", dpi=150, bbox_inches="tight")
plt.show()


## Cross-city summary: % under 2030 budget — ICE vs BEV

In [ ]:
rows = []
for city in ["Helsinki","Tampere","Turku","Oulu"]:
    ice = pd.read_parquet(f"./output/tour_car_{city.lower()}_decent_per_user.parquet")
    bev = bev_results[city]
    rows.append({
        "City": city,
        "ICE_under_budget_pct": (ice["decent_mobility_co2"] <= BUDGET_2030).mean()*100,
        "EV_under_budget_pct":  (bev["decent_mobility_co2"] <= BUDGET_2030).mean()*100,
    })

# Finland aggregate
all_ice = pd.concat([
    pd.read_parquet(f"./output/tour_car_{c.lower()}_decent_per_user.parquet")
    for c in ["Helsinki","Tampere","Turku","Oulu"]
])
all_bev = pd.concat([bev_results[c] for c in ["Helsinki","Tampere","Turku","Oulu"]])
rows.append({
    "City": "Overall",
    "ICE_under_budget_pct": (all_ice["decent_mobility_co2"] <= BUDGET_2030).mean()*100,
    "EV_under_budget_pct":  (all_bev["decent_mobility_co2"] <= BUDGET_2030).mean()*100,
})

summary = pd.DataFrame(rows)
summary


In [ ]:
order = ["Helsinki", "Tampere", "Turku", "Oulu", "Overall"]
summary["City"] = pd.Categorical(summary["City"], categories=order, ordered=True)
summary = summary.sort_values("City")

y = np.arange(len(summary))[::-1].astype(float)
finland_idx = list(summary["City"]).index("Overall")
y[finland_idx] -= 0.6

max_val = max(summary["ICE_under_budget_pct"].max(), summary["EV_under_budget_pct"].max())
x_max = np.ceil(max_val + 5)

fig, ax = plt.subplots(figsize=(10.5, 5.5))
bar_h    = 0.35
ice_color = "#b2182b"
ev_color  = "#2166ac"

ax.barh(y + bar_h/2, summary["ICE_under_budget_pct"], height=bar_h, color=ice_color, label="ICE")
ax.barh(y - bar_h/2, summary["EV_under_budget_pct"],  height=bar_h, color=ev_color,  label="BEV")

for i, row in enumerate(summary.itertuples()):
    ax.text(row.ICE_under_budget_pct + 0.6, y[i] + bar_h/2, f"{row.ICE_under_budget_pct:.1f}%", va="center", fontsize=11)
    ax.text(row.EV_under_budget_pct  + 0.6, y[i] - bar_h/2, f"{row.EV_under_budget_pct:.1f}%",  va="center", fontsize=11)

ax.set_yticks(y)
ax.set_yticklabels(summary["City"])
ax.set_xlim(0, x_max)
ax.set_xlabel("Share of users under carbon budget (<7 kg CO₂/week)")
ax.set_title("Carbon budget compliance: ICE vs BEV (zoomed scale)", pad=14)
ax.xaxis.grid(True, linestyle="--", alpha=0.25)
ax.yaxis.grid(False)
for s in ["top","right","left"]:
    ax.spines[s].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)

plt.tight_layout()
plt.savefig("./output/bev_vs_ice_cross_city.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# Build plotting dataframe
# =====================================================

cities = ["Helsinki", "Tampere", "Turku", "Oulu"]

rows = []

for city in cities:

    ice = pd.read_parquet(
        f"./output/tour_car_{city.lower()}_decent_per_user.parquet"
    )

    bev = bev_results[city]

    rows.extend([
        pd.DataFrame({
            "City": city,
            "Scenario": "ICE",
            "CO2": ice["decent_mobility_co2"] / 1000  # kg
        }),
        pd.DataFrame({
            "City": city,
            "Scenario": "BEV",
            "CO2": bev["decent_mobility_co2"] / 1000  # kg
        })
    ])

# =====================================================
# Overall aggregate
# =====================================================

all_ice = pd.concat([
    pd.read_parquet(
        f"./output/tour_car_{c.lower()}_decent_per_user.parquet"
    )
    for c in cities
])

all_bev = pd.concat([
    bev_results[c]
    for c in cities
])

rows.extend([
    pd.DataFrame({
        "City": "Overall",
        "Scenario": "ICE",
        "CO2": all_ice["decent_mobility_co2"] / 1000
    }),
    pd.DataFrame({
        "City": "Overall",
        "Scenario": "BEV",
        "CO2": all_bev["decent_mobility_co2"] / 1000
    })
])

plot_df = pd.concat(rows, ignore_index=True)

# =====================================================
# Settings
# =====================================================

BUDGET_KG = BUDGET_2030 / 1000

city_order = [
    "Helsinki",
    "Tampere",
    "Turku",
    "Oulu",
    "Overall"
]

positions = []
data = []
colors = []

x = 1

for city in city_order:

    ice_vals = plot_df[
        (plot_df["City"] == city) &
        (plot_df["Scenario"] == "ICE")
    ]["CO2"].dropna()

    bev_vals = plot_df[
        (plot_df["City"] == city) &
        (plot_df["Scenario"] == "BEV")
    ]["CO2"].dropna()

    data.extend([ice_vals, bev_vals])

    positions.extend([x, x + 0.8])

    if city == "Overall":
        colors.extend([
            "#7f0000",   # darker ICE
            "#08306b"    # darker BEV
        ])
    else:
        colors.extend([
            "#b2182b",
            "#2166ac"
        ])

    # extra gap before Overall
    if city == "Oulu":
        x += 3.8
    else:
        x += 2.5

# =====================================================
# Plot
# =====================================================

fig, ax = plt.subplots(figsize=(13, 6))

bp = ax.boxplot(
    data,
    positions=positions,
    widths=0.55,
    patch_artist=True,
    showfliers=False
)

# box colors
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)

# medians
for median in bp["medians"]:
    median.set_color("black")
    median.set_linewidth(1.8)

# =====================================================
# Carbon budget line
# =====================================================

ax.axhline(
    BUDGET_KG,
    color="black",
    linestyle="--",
    linewidth=1.5
)

ax.text(
    positions[-1] + 0.3,
    BUDGET_KG,
    "7 kg/week target",
    fontsize=10,
    va="bottom"
)

# =====================================================
# Y-axis truncation (95th percentile)
# =====================================================

upper = plot_df["CO2"].quantile(0.95)

ax.set_ylim(0, upper)

# =====================================================
# X labels
# =====================================================

centers = [
    np.mean([positions[i * 2], positions[i * 2 + 1]])
    for i in range(len(city_order))
]

ax.set_xticks(centers)
ax.set_xticklabels(city_order, fontsize=11)

# ICE / BEV labels
for i in range(0, len(positions), 2):

    ax.text(
        positions[i],
        -0.08,
        "ICE",
        ha="center",
        transform=ax.get_xaxis_transform(),
        fontsize=9,
        color="#b2182b"
    )

    ax.text(
        positions[i + 1],
        -0.08,
        "BEV",
        ha="center",
        transform=ax.get_xaxis_transform(),
        fontsize=9,
        color="#2166ac"
    )

# =====================================================
# Formatting
# =====================================================

ax.set_ylabel(
    "Weekly mobility carbon expenditure (kg CO₂e/week)",
    fontsize=12
)

ax.grid(
    axis="y",
    linestyle=":",
    alpha=0.4
)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

# note about truncation
ax.text(
    0.01,
    0.98,
    "Upper 5% of observations omitted for clarity",
    transform=ax.transAxes,
    fontsize=9,
    va="top"
)

plt.tight_layout()

# plt.savefig(
#     "./figures/boxplot_bev_vs_ice.png",
#     dpi=300,
#     bbox_inches="tight"
# )

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# Build plotting dataframe
# =====================================================

cities = ["Helsinki", "Tampere", "Turku", "Oulu"]

rows = []

for city in cities:

    ice = pd.read_parquet(
        f"./output/tour_car_{city.lower()}_decent_per_user.parquet"
    )

    bev = bev_results[city]

    rows.extend([
        pd.DataFrame({
            "City": city,
            "Scenario": "ICE",
            "CO2": ice["decent_mobility_co2"] / 1000  # kg
        }),
        pd.DataFrame({
            "City": city,
            "Scenario": "BEV",
            "CO2": bev["decent_mobility_co2"] / 1000  # kg
        })
    ])

# =====================================================
# Overall aggregate
# =====================================================

all_ice = pd.concat([
    pd.read_parquet(
        f"./output/tour_car_{c.lower()}_decent_per_user.parquet"
    )
    for c in cities
])

all_bev = pd.concat([
    bev_results[c]
    for c in cities
])

rows.extend([
    pd.DataFrame({
        "City": "Overall",
        "Scenario": "ICE",
        "CO2": all_ice["decent_mobility_co2"] / 1000
    }),
    pd.DataFrame({
        "City": "Overall",
        "Scenario": "BEV",
        "CO2": all_bev["decent_mobility_co2"] / 1000
    })
])

plot_df = pd.concat(rows, ignore_index=True)

# =====================================================
# Settings
# =====================================================

BUDGET_KG = BUDGET_2030 / 1000

city_order = [
    "Helsinki",
    "Tampere",
    "Turku",
    "Oulu",
    "Overall"
]

positions = []
data = []
colors = []

x = 1

for city in city_order:

    ice_vals = plot_df[
        (plot_df["City"] == city) &
        (plot_df["Scenario"] == "ICE")
    ]["CO2"].dropna()

    bev_vals = plot_df[
        (plot_df["City"] == city) &
        (plot_df["Scenario"] == "BEV")
    ]["CO2"].dropna()

    data.extend([ice_vals, bev_vals])
    positions.extend([x, x + 0.8])

    if city == "Overall":
        colors.extend(["#7f0000", "#08306b"])
    else:
        colors.extend(["#b2182b", "#2166ac"])

    if city == "Oulu":
        x += 3.0
    else:
        x += 2.5

# =====================================================
# Plot
# =====================================================

fig, ax = plt.subplots(figsize=(13, 6))

bp = ax.boxplot(
    data,
    positions=positions,
    widths=0.55,
    patch_artist=True,
    showfliers=False
)

# box colors
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)

# medians
for median in bp["medians"]:
    median.set_color("black")
    median.set_linewidth(1.8)

# =====================================================
# Carbon budget line
# =====================================================

ax.axhline(
    BUDGET_KG,
    color="black",
    linestyle="--",
    linewidth=1.5
)

# Put label OUTSIDE plot, right margin
ax.text(
    1.01,
    BUDGET_KG,
    "7 kg CO₂e/week\n target",
    transform=ax.get_yaxis_transform(),
    fontsize=10,
    va="center",
    ha="left",
    clip_on=False
)

# =====================================================
# X labels
# =====================================================

centers = [
    np.mean([positions[i * 2], positions[i * 2 + 1]])
    for i in range(len(city_order))
]

ax.set_xticks(centers)
ax.set_xticklabels(city_order, fontsize=11)

# ICE / BEV labels
for i in range(0, len(positions), 2):

    ax.text(
        positions[i],
        -0.08,
        "ICE",
        ha="center",
        transform=ax.get_xaxis_transform(),
        fontsize=9,
        color="#b2182b"
    )

    ax.text(
        positions[i + 1],
        -0.08,
        "BEV",
        ha="center",
        transform=ax.get_xaxis_transform(),
        fontsize=9,
        color="#2166ac"
    )

# =====================================================
# Formatting
# =====================================================

ax.set_ylabel(
    "Weekly carbon expenditure (kg CO₂e/week)",
    fontsize=12
)

ax.grid(axis="y", linestyle=":", alpha=0.4)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# Build plotting dataframe
# =====================================================

cities = ["Helsinki", "Turku", "Tampere", "Oulu"]

rows = []

for city in cities:

    ice = pd.read_parquet(
        f"./output/tour_car_{city.lower()}_decent_per_user.parquet"
    )

    bev = bev_results[city]

    rows.extend([
        pd.DataFrame({
            "City": city,
            "Scenario": "ICE",
            "CO2": ice["decent_mobility_co2"] / 1000  # kg
        }),
        pd.DataFrame({
            "City": city,
            "Scenario": "BEV",
            "CO2": bev["decent_mobility_co2"] / 1000  # kg
        })
    ])

# =====================================================
# Overall aggregate
# =====================================================

all_ice = pd.concat([
    pd.read_parquet(
        f"./output/tour_car_{c.lower()}_decent_per_user.parquet"
    )
    for c in cities
])

all_bev = pd.concat([
    bev_results[c]
    for c in cities
])

rows.extend([
    pd.DataFrame({
        "City": "Overall",
        "Scenario": "ICE",
        "CO2": all_ice["decent_mobility_co2"] / 1000
    }),
    pd.DataFrame({
        "City": "Overall",
        "Scenario": "BEV",
        "CO2": all_bev["decent_mobility_co2"] / 1000
    })
])

plot_df = pd.concat(rows, ignore_index=True)

# =====================================================
# CAP ICE VALUES (95th percentile)
# =====================================================

ice_95_percentile = plot_df[plot_df["Scenario"] == "ICE"]["CO2"].quantile(0.95)

plot_df.loc[
    (plot_df["Scenario"] == "ICE") & (plot_df["CO2"] > ice_95_percentile),
    "CO2"
] = ice_95_percentile

# =====================================================
# Settings
# =====================================================

BUDGET_KG = BUDGET_2030 / 1000

city_order = [
    "Helsinki",
    "Turku",
    "Tampere",
    "Oulu",
    "Overall"
]

positions = []
data = []
colors = []

x = 1

for city in city_order:

    ice_vals = plot_df[
        (plot_df["City"] == city) &
        (plot_df["Scenario"] == "ICE")
    ]["CO2"].dropna()

    bev_vals = plot_df[
        (plot_df["City"] == city) &
        (plot_df["Scenario"] == "BEV")
    ]["CO2"].dropna()

    data.extend([ice_vals, bev_vals])
    positions.extend([x, x + 0.8])

    # --------- UNIFORM COLORS ---------
    colors.extend(["#b2182b", "#2166ac"])  # Same red and blue for ALL cities

    if city == "Oulu":
        x += 3.0
    else:
        x += 2.5

# =====================================================
# Plot
# =====================================================

fig, ax = plt.subplots(figsize=(13, 6))

bp = ax.boxplot(
    data,
    positions=positions,
    widths=0.55,
    patch_artist=True,
    showfliers=False
)

# box colors
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)

# medians
for median in bp["medians"]:
    median.set_color("black")
    median.set_linewidth(1.8)

# =====================================================
# Carbon budget line
# =====================================================

ax.axhline(
    BUDGET_KG,
    color="black",
    linestyle="--",
    linewidth=1.5
)

# Put label OUTSIDE plot, right margin
ax.text(
    1.01,
    BUDGET_KG,
    "7 kg CO₂e/week\n target",
    transform=ax.get_yaxis_transform(),
    fontsize=10,
    va="center",
    ha="left",
    clip_on=False
)

# =====================================================
# X labels
# =====================================================

centers = [
    np.mean([positions[i * 2], positions[i * 2 + 1]])
    for i in range(len(city_order))
]

ax.set_xticks(centers)
ax.set_xticklabels(city_order, fontsize=11)

# ICE / BEV labels
for i in range(0, len(positions), 2):

    ax.text(
        positions[i],
        -0.08,
        "ICE",
        ha="center",
        transform=ax.get_xaxis_transform(),
        fontsize=9,
        color="#b2182b"
    )

    ax.text(
        positions[i + 1],
        -0.08,
        "BEV",
        ha="center",
        transform=ax.get_xaxis_transform(),
        fontsize=9,
        color="#2166ac"
    )

# =====================================================
# Formatting
# =====================================================

ax.set_ylabel(
    "Weekly carbon expenditure (kg CO₂e/week)",
    fontsize=12
)

ax.grid(axis="y", linestyle=":", alpha=0.4)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(
    "./output/co2_boxplot_BEV_ICE.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# Build plot data (Overall last)
# =====================================================

cities = ["Helsinki", "Turku", "Tampere", "Oulu"]

rows = []

# -------------------------
# City-level data first
# -------------------------
for city in cities:

    ice = pd.read_parquet(
        f"./output/tour_car_{city.lower()}_decent_per_user.parquet"
    )

    bev = bev_results[city]

    rows.append(pd.DataFrame({
        "City": city,
        "Scenario": "ICE",
        "CO2": ice["decent_mobility_co2"] / 1000
    }))

    rows.append(pd.DataFrame({
        "City": city,
        "Scenario": "BEV",
        "CO2": bev["decent_mobility_co2"] / 1000
    }))

# -------------------------
# Overall LAST (IMPORTANT FIX)
# -------------------------
all_ice = pd.concat([
    pd.read_parquet(f"./output/tour_car_{c.lower()}_decent_per_user.parquet")
    for c in cities
])

all_bev = pd.concat([
    bev_results[c]
    for c in cities
])

rows.append(pd.DataFrame({
    "City": "Overall",
    "Scenario": "ICE",
    "CO2": all_ice["decent_mobility_co2"] / 1000
}))

rows.append(pd.DataFrame({
    "City": "Overall",
    "Scenario": "BEV",
    "CO2": all_bev["decent_mobility_co2"] / 1000
}))

plot_df = pd.concat(rows, ignore_index=True)

# =====================================================
# ORDER ENFORCEMENT (FINAL SAFETY NET)
# =====================================================

city_order = ["Helsinki", "Turku", "Tampere", "Oulu", "Overall"]
scenarios = ["ICE", "BEV"]

plot_df["City"] = pd.Categorical(
    plot_df["City"],
    categories=city_order,
    ordered=True
)

plot_df["Scenario"] = pd.Categorical(
    plot_df["Scenario"],
    categories=scenarios,
    ordered=True
)

# =====================================================
# COMPUTE SUMMARY
# =====================================================

results = []

for city in city_order:
    for scen in scenarios:

        subset = plot_df[
            (plot_df["City"] == city) &
            (plot_df["Scenario"] == scen)
        ]["CO2"].dropna()

        if len(subset) == 0:
            continue

        under = (subset <= BUDGET_KG).mean() * 100
        over = 100 - under

        results.append({
            "City": city,
            "Scenario": scen,
            "Under": under,
            "Over": over
        })

summary = pd.DataFrame(results)

# =====================================================
# PLOT
# =====================================================

fig, ax = plt.subplots(figsize=(12, 6))

bar_h = 0.75
y = 0

yticks = []
yticklabels = []

for idx, (_, row) in enumerate(summary.iterrows()):

    city = row["City"]
    scen = row["Scenario"]
    under = row["Under"]
    over = row["Over"]

    color_main = "#b2182b" if scen == "ICE" else "#2166ac"

    # -------------------------
    # UNDER budget
    # -------------------------
    ax.barh(
        y,
        under,
        height=bar_h,
        color=color_main,
        alpha=0.9
    )

    ax.text(
        under / 2,
        y,
        f"{under:.0f}%",
        va="center",
        ha="center",
        fontsize=11,
        color="white",
        fontweight="bold"
    )

    # -------------------------
    # OVER budget
    # -------------------------
    ax.barh(
        y,
        over,
        left=under,
        height=bar_h,
        color="#d9d9d9",
        alpha=1.0
    )

    ax.text(
        under + over / 2,
        y,
        f"{over:.0f}%",
        va="center",
        ha="center",
        fontsize=10,
        color="black"
    )

    yticks.append(y)
    yticklabels.append(f"{city} · {scen}")

    y += 1

    # -------------------------
    # Spacing logic
    # -------------------------
    if scen == "BEV":
        # Extra gap BEFORE "Overall" (after Oulu)
        if city == "Oulu":
            y += 1.2  # Extra space before Overall
        else:
            y += 0.8  # Regular spacing between city pairs

# =====================================================
# FORMATTING
# =====================================================

ax.set_yticks(yticks)
ax.set_yticklabels(yticklabels, fontsize=11)

ax.invert_yaxis()  # Flip so Oulu is on top, Overall on bottom

ax.set_xlim(0, 100)

ax.set_xlabel(
    "Share of users under CO₂ budget (%)",
    fontsize=12
)

ax.grid(axis="x", linestyle=":", alpha=0.35)

# =====================================================
# LEGEND
# =====================================================

from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="#b2182b", alpha=0.9, label="ICE (Under Budget)"),
    Patch(facecolor="#2166ac", alpha=0.9, label="BEV (Under Budget)"),
    Patch(facecolor="#d9d9d9", alpha=1.0, label="Over Budget")
]

ax.legend(
    handles=legend_elements,
    loc="lower right",
    fontsize=10,
    frameon=True,
    fancybox=False,
    edgecolor="black",
    framealpha=0.95
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()

plt.savefig(
    "./output/co2_bars_BEV_ICE.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# Build plotting dataframe
# =====================================================

cities = ["Helsinki", "Turku", "Tampere", "Oulu"]

rows = []

for city in cities:

    ice = pd.read_parquet(
        f"./output/tour_car_{city.lower()}_decent_per_user.parquet"
    )

    bev = bev_results[city]

    rows.extend([
        pd.DataFrame({
            "City": city,
            "Scenario": "ICE",
            "CO2": ice["decent_mobility_co2"] / 1000  # kg
        }),
        pd.DataFrame({
            "City": city,
            "Scenario": "BEV",
            "CO2": bev["decent_mobility_co2"] / 1000  # kg
        })
    ])

# =====================================================
# Overall aggregate
# =====================================================

all_ice = pd.concat([
    pd.read_parquet(
        f"./output/tour_car_{c.lower()}_decent_per_user.parquet"
    )
    for c in cities
])

all_bev = pd.concat([
    bev_results[c]
    for c in cities
])

rows.extend([
    pd.DataFrame({
        "City": "Overall",
        "Scenario": "ICE",
        "CO2": all_ice["decent_mobility_co2"] / 1000
    }),
    pd.DataFrame({
        "City": "Overall",
        "Scenario": "BEV",
        "CO2": all_bev["decent_mobility_co2"] / 1000
    })
])

plot_df = pd.concat(rows, ignore_index=True)

# =====================================================
# CAP ICE VALUES (95th percentile)
# =====================================================

ice_95_percentile = plot_df[plot_df["Scenario"] == "ICE"]["CO2"].quantile(0.95)

plot_df.loc[
    (plot_df["Scenario"] == "ICE") & (plot_df["CO2"] > ice_95_percentile),
    "CO2"
] = ice_95_percentile

# =====================================================
# Settings
# =====================================================

BUDGET_KG = BUDGET_2030 / 1000

city_order = [
    "Helsinki",
    "Turku",
    "Tampere",
    "Oulu",
    "Overall"
]

# --------- COLORS (from project palette) ---------
ICE_COLOR = "#9A2588"   # C2 - same plum/magenta used for "Car" elsewhere
BEV_COLOR = "#4385BE"   # A3/A5 - blue, consistent with "cleaner/under budget" elsewhere

positions = []
data = []
colors = []

x = 1

for city in city_order:

    ice_vals = plot_df[
        (plot_df["City"] == city) &
        (plot_df["Scenario"] == "ICE")
    ]["CO2"].dropna()

    bev_vals = plot_df[
        (plot_df["City"] == city) &
        (plot_df["Scenario"] == "BEV")
    ]["CO2"].dropna()

    data.extend([ice_vals, bev_vals])
    positions.extend([x, x + 0.8])

    colors.extend([ICE_COLOR, BEV_COLOR])

    if city == "Oulu":
        x += 3.0
    else:
        x += 2.5

# =====================================================
# Plot
# =====================================================

fig, ax = plt.subplots(figsize=(13, 6))

bp = ax.boxplot(
    data,
    positions=positions,
    widths=0.55,
    patch_artist=True,
    showfliers=False
)

# box colors
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)

# medians
for median in bp["medians"]:
    median.set_color("white")
    median.set_linewidth(1.8)

# =====================================================
# Carbon budget line
# =====================================================

ax.axhline(
    BUDGET_KG,
    color="black",
    linestyle="--",
    linewidth=1.5
)

# Put label OUTSIDE plot, right margin
ax.text(
    1.01,
    BUDGET_KG,
    "7 kg CO₂e/week\n target",
    transform=ax.get_yaxis_transform(),
    fontsize=11,
    va="center",
    ha="left",
    clip_on=False
)

# =====================================================
# X labels
# =====================================================

centers = [
    np.mean([positions[i * 2], positions[i * 2 + 1]])
    for i in range(len(city_order))
]

ax.set_xticks(centers)
ax.set_xticklabels(city_order, fontsize=15)

# ICE / BEV labels
for i in range(0, len(positions), 2):

    ax.text(
        positions[i],
        -0.08,
        "ICE",
        ha="center",
        transform=ax.get_xaxis_transform(),
        fontsize=12,
        color=ICE_COLOR
    )

    ax.text(
        positions[i + 1],
        -0.08,
        "BEV",
        ha="center",
        transform=ax.get_xaxis_transform(),
        fontsize=12,
        color=BEV_COLOR
    )

# =====================================================
# Formatting
# =====================================================

ax.set_ylabel(
    "Weekly carbon expenditure (kg CO₂e/week)",
    fontsize=16
)

ax.grid(axis="y", linestyle=":", alpha=0.4)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(
    "./output/co2_boxplot_BEV_ICE.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# Build plot data (Overall last)
# =====================================================

cities = ["Helsinki", "Turku", "Tampere", "Oulu"]

rows = []

# -------------------------
# City-level data first
# -------------------------
for city in cities:

    ice = pd.read_parquet(
        f"./output/tour_car_{city.lower()}_decent_per_user.parquet"
    )

    bev = bev_results[city]

    rows.append(pd.DataFrame({
        "City": city,
        "Scenario": "ICE",
        "CO2": ice["decent_mobility_co2"] / 1000
    }))

    rows.append(pd.DataFrame({
        "City": city,
        "Scenario": "BEV",
        "CO2": bev["decent_mobility_co2"] / 1000
    }))

# -------------------------
# Overall LAST (IMPORTANT FIX)
# -------------------------
all_ice = pd.concat([
    pd.read_parquet(f"./output/tour_car_{c.lower()}_decent_per_user.parquet")
    for c in cities
])

all_bev = pd.concat([
    bev_results[c]
    for c in cities
])

rows.append(pd.DataFrame({
    "City": "Overall",
    "Scenario": "ICE",
    "CO2": all_ice["decent_mobility_co2"] / 1000
}))

rows.append(pd.DataFrame({
    "City": "Overall",
    "Scenario": "BEV",
    "CO2": all_bev["decent_mobility_co2"] / 1000
}))

plot_df = pd.concat(rows, ignore_index=True)

# =====================================================
# ORDER ENFORCEMENT (FINAL SAFETY NET)
# =====================================================

city_order = ["Helsinki", "Turku", "Tampere", "Oulu", "Overall"]
scenarios = ["ICE", "BEV"]

plot_df["City"] = pd.Categorical(
    plot_df["City"],
    categories=city_order,
    ordered=True
)

plot_df["Scenario"] = pd.Categorical(
    plot_df["Scenario"],
    categories=scenarios,
    ordered=True
)

# =====================================================
# COMPUTE SUMMARY
# =====================================================

results = []

for city in city_order:
    for scen in scenarios:

        subset = plot_df[
            (plot_df["City"] == city) &
            (plot_df["Scenario"] == scen)
        ]["CO2"].dropna()

        if len(subset) == 0:
            continue

        under = (subset <= BUDGET_KG).mean() * 100
        over = 100 - under

        results.append({
            "City": city,
            "Scenario": scen,
            "Under": under,
            "Over": over
        })

summary = pd.DataFrame(results)

# =====================================================
# COLORS (from project palette)
# =====================================================

ICE_COLOR = "#9A2588"   # C2 - same plum/magenta used for "Car" elsewhere
BEV_COLOR = "#4385BE"   # A3/A5 - blue, consistent with "cleaner/under budget" elsewhere
OVER_BUDGET_COLOR = "#D9D9D9"  # unchanged neutral grey

# =====================================================
# PLOT
# =====================================================

fig, ax = plt.subplots(figsize=(12, 6))

bar_h = 0.75
y = 0

yticks = []
yticklabels = []

for idx, (_, row) in enumerate(summary.iterrows()):

    city = row["City"]
    scen = row["Scenario"]
    under = row["Under"]
    over = row["Over"]

    color_main = ICE_COLOR if scen == "ICE" else BEV_COLOR

    # -------------------------
    # UNDER budget
    # -------------------------
    ax.barh(
        y,
        under,
        height=bar_h,
        color=color_main,
        alpha=0.9
    )

    ax.text(
        under / 2,
        y,
        f"{under:.0f}%",
        va="center",
        ha="center",
        fontsize=13,
        color="white",
        fontweight="bold"
    )

    # -------------------------
    # OVER budget
    # -------------------------
    ax.barh(
        y,
        over,
        left=under,
        height=bar_h,
        color=OVER_BUDGET_COLOR,
        alpha=1.0
    )

    ax.text(
        under + over / 2,
        y,
        f"{over:.0f}%",
        va="center",
        ha="center",
        fontsize=12,
        color="black"
    )

    yticks.append(y)
    yticklabels.append(f"{city} · {scen}")

    y += 1

    # -------------------------
    # Spacing logic
    # -------------------------
    if scen == "BEV":
        # Extra gap BEFORE "Overall" (after Oulu)
        if city == "Oulu":
            y += 1.2  # Extra space before Overall
        else:
            y += 0.8  # Regular spacing between city pairs

# =====================================================
# FORMATTING
# =====================================================

ax.set_yticks(yticks)
ax.set_yticklabels(yticklabels, fontsize=11)

ax.invert_yaxis()  # Flip so Oulu is on top, Overall on bottom

ax.set_xlim(0, 100)

ax.set_xlabel(
    "Share of users under CO₂ budget (%)",
    fontsize=15
)

ax.grid(axis="x", linestyle=":", alpha=0.35)

# =====================================================
# LEGEND
# =====================================================

from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor=ICE_COLOR, alpha=0.9, label="ICE (Under Budget)"),
    Patch(facecolor=BEV_COLOR, alpha=0.9, label="BEV (Under Budget)"),
    Patch(facecolor=OVER_BUDGET_COLOR, alpha=1.0, label="Over Budget")
]

ax.legend(
    handles=legend_elements,
    loc="lower right",
    fontsize=13,
    frameon=True,
    fancybox=False,
    edgecolor="black",
    framealpha=0.95
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()

plt.savefig(
    "./output/co2_bars_BEV_ICE.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# Build plot data (Overall last)
# =====================================================
cities = ["Helsinki", "Turku", "Tampere", "Oulu"]
rows = []

# -------------------------
# City-level data first
# -------------------------
for city in cities:
    ice = pd.read_parquet(
        f"./output/tour_car_{city.lower()}_decent_per_user.parquet"
    )
    bev = bev_results[city]
    rows.append(pd.DataFrame({
        "City": city,
        "Scenario": "ICE",
        "CO2": ice["decent_mobility_co2"] / 1000
    }))
    rows.append(pd.DataFrame({
        "City": city,
        "Scenario": "BEV",
        "CO2": bev["decent_mobility_co2"] / 1000
    }))

# -------------------------
# Overall LAST (IMPORTANT FIX)
# -------------------------
all_ice = pd.concat([
    pd.read_parquet(f"./output/tour_car_{c.lower()}_decent_per_user.parquet")
    for c in cities
])
all_bev = pd.concat([
    bev_results[c]
    for c in cities
])
rows.append(pd.DataFrame({
    "City": "Overall",
    "Scenario": "ICE",
    "CO2": all_ice["decent_mobility_co2"] / 1000
}))
rows.append(pd.DataFrame({
    "City": "Overall",
    "Scenario": "BEV",
    "CO2": all_bev["decent_mobility_co2"] / 1000
}))

plot_df = pd.concat(rows, ignore_index=True)

# =====================================================
# ORDER ENFORCEMENT (FINAL SAFETY NET)
# =====================================================
city_order = ["Helsinki", "Turku", "Tampere", "Oulu", "Overall"]
scenarios = ["ICE", "BEV"]

plot_df["City"] = pd.Categorical(
    plot_df["City"],
    categories=city_order,
    ordered=True
)
plot_df["Scenario"] = pd.Categorical(
    plot_df["Scenario"],
    categories=scenarios,
    ordered=True
)

# =====================================================
# COMPUTE SUMMARY
# =====================================================
results = []
for city in city_order:
    for scen in scenarios:
        subset = plot_df[
            (plot_df["City"] == city) &
            (plot_df["Scenario"] == scen)
        ]["CO2"].dropna()
        if len(subset) == 0:
            continue
        under = (subset <= BUDGET_KG).mean() * 100
        over = 100 - under
        results.append({
            "City": city,
            "Scenario": scen,
            "Under": under,
            "Over": over
        })
summary = pd.DataFrame(results)

# =====================================================
# COLORS (from project palette)
# =====================================================
ICE_COLOR = "#9A2588"          # C2 - plum/magenta
BEV_COLOR = "#4385BE"          # A3/A5 - blue
OVER_BUDGET_COLOR = "#D9D9D9"  # neutral grey

# =====================================================
# PLOT
# =====================================================
fig, ax = plt.subplots(figsize=(12, 6))

bar_h = 0.75
y = 0
yticks = []
yticklabels = []

for idx, (_, row) in enumerate(summary.iterrows()):
    city = row["City"]
    scen = row["Scenario"]
    under = row["Under"]
    over = row["Over"]

    color_main = ICE_COLOR if scen == "ICE" else BEV_COLOR

    # -------------------------
    # UNDER budget
    # -------------------------
    ax.barh(
        y,
        under,
        height=bar_h,
        color=color_main,
        alpha=0.9
    )
    ax.text(
        under / 2,
        y,
        f"{under:.0f}%",
        va="center",
        ha="center",
        fontsize=13,
        color="white",
        fontweight="bold"
    )

    # -------------------------
    # OVER budget
    # -------------------------
    ax.barh(
        y,
        over,
        left=under,
        height=bar_h,
        color=OVER_BUDGET_COLOR,
        alpha=1.0
    )
    ax.text(
        under + over / 2,
        y,
        f"{over:.0f}%",
        va="center",
        ha="center",
        fontsize=13,
        color="black"
    )

    yticks.append(y)
    yticklabels.append(f"{city} · {scen}")
    y += 1

    # -------------------------
    # Spacing logic
    # -------------------------
    if scen == "BEV":
        if city == "Oulu":
            y += 1.2   # Extra space before Overall
        else:
            y += 0.8   # Regular spacing between city pairs

# =====================================================
# FORMATTING
# =====================================================
ax.set_yticks(yticks)
ax.set_yticklabels(yticklabels, fontsize=14)  # ← increased from 11

ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel(
    "Share of users under CO₂ budget (%)",
    fontsize=15
)
ax.grid(axis="x", linestyle=":", alpha=0.35)

# =====================================================
# LEGEND
# =====================================================
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor=ICE_COLOR, alpha=0.9, label="ICE (Under Budget)"),
    Patch(facecolor=BEV_COLOR, alpha=0.9, label="BEV (Under Budget)"),
    Patch(facecolor=OVER_BUDGET_COLOR, alpha=1.0, label="Over Budget")
]
ax.legend(
    handles=legend_elements,
    loc="lower right",
    fontsize=13,
    frameon=True,
    fancybox=False,
    edgecolor="black",
    framealpha=0.95
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(
    "./output/co2_bars_BEV_ICE.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()

In [ ]:
"""
Figure 2 — Helsinki home-to-work OD flow map
- Gradient arc: RED (origin/home) → BLUE (destination/work)
- No filter boundary drawn
- No separate home/work dot colors — single dot per hexagon,
  radius scaled by TOTAL trips (originated + received)
- Line width strongly differentiated by flow volume (log scale)
"""

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import contextily as ctx; import basemaps
from h3 import h3
from pyproj import Transformer

TR = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

# ── CONFIG ──────────────────────────────────────────────────────────────────
USERS_PATH = "scratch/data/users_and_stays_3months.parquet"

MIN_FLOW       = 1
TOP_N_FLOWS    = 1000
TOP_N_DOTS     = 500
DOT_H3_LEVEL   = 8
ARC_HEIGHT     = 0.32
ORIGIN_COLOR   = np.array([178/255, 24/255, 43/255, 1.0])   # red  — origin (home)
DEST_COLOR     = np.array([33/255, 102/255, 172/255, 1.0])  # blue — destination (work)
DOT_COLOR      = "#404040"
LINE_WIDTH_MIN = 0.6
LINE_WIDTH_MAX = 9.0
BASE_ALPHA     = 0.75


def h3_to_xy(h3_id):
    lat, lon = h3.h3_to_geo(h3_id)
    return TR.transform(lon, lat)


def build_arc_xy(x1, y1, x2, y2, height=ARC_HEIGHT, n=80):
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    dx, dy = x2 - x1, y2 - y1
    cx = mx - dy * height
    cy = my + dx * height
    t  = np.linspace(0, 1, n)
    xs = (1-t)**2 * x1 + 2*(1-t)*t * cx + t**2 * x2
    ys = (1-t)**2 * y1 + 2*(1-t)*t * cy + t**2 * y2
    return xs, ys


def draw_gradient_arc(ax, xs, ys, lw, alpha):
    """Draw arc as segments graduating ORIGIN(red) -> DEST(blue)."""
    n = len(xs)
    for i in range(n - 1):
        t  = i / (n - 2)
        c  = (1 - t) * ORIGIN_COLOR + t * DEST_COLOR
        c[3] = alpha
        ax.plot(xs[i:i+2], ys[i:i+2], color=c, lw=lw,
                solid_capstyle="round", zorder=2)


import geopandas as gpd
from shapely.geometry import Point

FILTER_GEOJSON = "./data/helsinki_filter_map.geojson"

# ── LOAD DATA ───────────────────────────────────────────────────────────────
print("Loading Helsinki users...")
users = pd.read_parquet(USERS_PATH)
users = users[users["home_gid9"].notna() & users["work_gid9"].notna()]

home_work = (
    users[["user_id", "home_gid9", "work_gid9"]]
    .drop_duplicates(subset="user_id")
    .copy()
)
home_work = home_work[home_work["home_gid9"] != home_work["work_gid9"]]

# ── SPATIAL FILTER: keep only rows where BOTH home and work fall
#    inside the city boundary geojson. Boundary is used purely as a
#    filter — it is never drawn on the map.
boundary = gpd.read_file(FILTER_GEOJSON).to_crs("EPSG:4326").union_all()

unique_hexes = pd.unique(
    pd.concat([home_work["home_gid9"], home_work["work_gid9"]]).values
)
inside_boundary = {}
for h in unique_hexes:
    lat, lon = h3.h3_to_geo(h)
    inside_boundary[h] = boundary.contains(Point(lon, lat))

home_work["home_inside"] = home_work["home_gid9"].map(inside_boundary)
home_work["work_inside"] = home_work["work_gid9"].map(inside_boundary)
home_work = home_work[home_work["home_inside"] & home_work["work_inside"]].drop(
    columns=["home_inside", "work_inside"]
)
print(f"Home-work pairs after boundary filter: {len(home_work)}")

flows = (
    home_work
    .groupby(["home_gid9", "work_gid9"])
    .size()
    .reset_index(name="n_users")
    .query("n_users >= @MIN_FLOW")
    .sort_values("n_users", ascending=False)
    .head(TOP_N_FLOWS)
)
print(f"Flows kept: {len(flows)}  (max volume: {flows['n_users'].max()})")

# ── TOTAL TRIPS PER HEXAGON (originated + received) — MERGED TO LEVEL 8 ────
# Dots are aggregated to coarser H3 level 8 cells (fewer, larger, more
# legible markers) and only the top 500 by total volume are drawn.
home_work["home_h8"] = home_work["home_gid9"].map(lambda h: h3.h3_to_parent(h, DOT_H3_LEVEL))
home_work["work_h8"] = home_work["work_gid9"].map(lambda h: h3.h3_to_parent(h, DOT_H3_LEVEL))

orig_counts = home_work.groupby("home_h8").size().reset_index(name="n_orig").rename(columns={"home_h8": "hex"})
dest_counts = home_work.groupby("work_h8").size().reset_index(name="n_dest").rename(columns={"work_h8": "hex"})
total_counts = (
    orig_counts.merge(dest_counts, on="hex", how="outer")
    .fillna(0)
)
total_counts["n_total"] = total_counts["n_orig"] + total_counts["n_dest"]
total_counts = total_counts.sort_values("n_total", ascending=False).head(TOP_N_DOTS)

# ── CENTROIDS — level-9 hexes for flow arcs, level-8 hexes for dots ─────────
all_hexes_full = pd.unique(
    pd.concat([home_work["home_gid9"], home_work["work_gid9"]]).values
)
centroids = {h: h3_to_xy(h) for h in all_hexes_full}

dot_centroids = {h: h3_to_xy(h) for h in total_counts["hex"]}

# ── PLOT ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 11))

all_xy = np.array(list(centroids.values()))
xmin, ymin = all_xy.min(axis=0)
xmax, ymax = all_xy.max(axis=0)
pad_x = (xmax - xmin) * 0.06
pad_y = (ymax - ymin) * 0.06
ax.set_xlim(xmin - pad_x, xmax + pad_x)
ax.set_ylim(ymin - pad_y, ymax + pad_y)

try:
    ctx.add_basemap(ax, crs="EPSG:3857", source=basemaps.POSITRON,
                    zoom="auto", attribution=False)
except Exception:
    ax.set_facecolor("#f0f0f0")

# Log-scaled line width for strong visual differentiation
flows["log_n"] = np.log1p(flows["n_users"])
log_min, log_max = flows["log_n"].min(), flows["log_n"].max()

for _, row in flows.iterrows():
    h_hex, w_hex = row["home_gid9"], row["work_gid9"]
    x1, y1 = centroids[h_hex]
    x2, y2 = centroids[w_hex]

    t_val = (row["log_n"] - log_min) / (log_max - log_min + 1e-9)
    lw    = LINE_WIDTH_MIN + t_val * (LINE_WIDTH_MAX - LINE_WIDTH_MIN)
    alpha = BASE_ALPHA * (0.35 + 0.65 * t_val)

    xs, ys = build_arc_xy(x1, y1, x2, y2)
    draw_gradient_arc(ax, xs, ys, lw, alpha)

# Dots: single color, radius = total trips (sqrt scale for area accuracy)
sizes = np.sqrt(total_counts["n_total"].values)
sizes_scaled = np.clip(sizes / sizes.max() * 220, 8, 220)
xy_dots = np.array([dot_centroids[h] for h in total_counts["hex"]])

ax.scatter(xy_dots[:, 0], xy_dots[:, 1], s=sizes_scaled, c=DOT_COLOR,
           alpha=0.85, linewidths=0.4, edgecolors="white", zorder=4)

# Legend
legend_elements = [
    Line2D([0], [0], color=ORIGIN_COLOR[:3], lw=4, label="Origin (home)"),
    Line2D([0], [0], color=DEST_COLOR[:3], lw=4, label="Destination (work)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=DOT_COLOR,
           markersize=10, label="Hexagon (size = total trips)"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=10, frameon=False)

ax.set_title("Helsinki — home-to-work commuting flows", fontsize=15, pad=10)
ax.set_axis_off()

plt.tight_layout()
plt.savefig("./output/fig2_od_flows_helsinki.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: ./output/fig2_od_flows_helsinki.png")
        

In [ ]:
"""
Figure 2 — Helsinki home-to-work OD flow map using datashader's hammer_bundle
(force-directed style edge bundling, fast vectorised implementation).

- Edges bundled to reveal corridor structure
- Gradient color along path: RED (origin/home) -> BLUE (destination/work)
- Dots aggregated to H3 level 8, top 500 by total volume
- Spatial filter: only OD pairs with both endpoints inside city boundary
  (boundary used only as filter, never drawn)
"""

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection
import contextily as ctx; import basemaps
from h3 import h3
from pyproj import Transformer
from shapely.geometry import Point
from datashader.bundling import hammer_bundle

TR = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

# ── CONFIG ──────────────────────────────────────────────────────────────────
USERS_PATH      = "scratch/data/users_and_stays_3months.parquet"
FILTER_GEOJSON  = "./data/helsinki_filter_map.geojson"

MIN_FLOW        = 1
TOP_N_FLOWS     = 1000
TOP_N_DOTS      = 500
DOT_H3_LEVEL    = 8

ORIGIN_COLOR    = np.array([178/255, 24/255, 43/255])    # red  — origin (home)
DEST_COLOR      = np.array([33/255, 102/255, 172/255])   # blue — destination (work)
DOT_COLOR       = "#2b2b2b"
LINE_WIDTH_MIN  = 0.35
LINE_WIDTH_MAX  = 2.6
BASE_ALPHA      = 0.55

# hammer_bundle parameters
HB_DECAY        = 0.6     # higher = more bundling
HB_ITERATIONS   = 6
HB_INITIAL_BW   = 0.05
HB_TENSION      = 0.4


def h3_to_xy(h3_id):
    lat, lon = h3.h3_to_geo(h3_id)
    return TR.transform(lon, lat)


# ── LOAD DATA ───────────────────────────────────────────────────────────────
print("Loading Helsinki users...")
users = pd.read_parquet(USERS_PATH)
users = users[users["home_gid9"].notna() & users["work_gid9"].notna()]

home_work = (
    users[["user_id", "home_gid9", "work_gid9"]]
    .drop_duplicates(subset="user_id")
    .copy()
)
home_work = home_work[home_work["home_gid9"] != home_work["work_gid9"]]

# Spatial filter: keep only rows where BOTH home and work fall inside boundary
boundary = gpd.read_file(FILTER_GEOJSON).to_crs("EPSG:4326").union_all()
unique_hexes = pd.unique(pd.concat([home_work["home_gid9"], home_work["work_gid9"]]).values)
inside_boundary = {}
for h in unique_hexes:
    lat, lon = h3.h3_to_geo(h)
    inside_boundary[h] = boundary.contains(Point(lon, lat))

home_work["home_inside"] = home_work["home_gid9"].map(inside_boundary)
home_work["work_inside"] = home_work["work_gid9"].map(inside_boundary)
home_work = home_work[home_work["home_inside"] & home_work["work_inside"]].drop(
    columns=["home_inside", "work_inside"]
)
print(f"Home-work pairs after boundary filter: {len(home_work)}")

flows = (
    home_work
    .groupby(["home_gid9", "work_gid9"])
    .size()
    .reset_index(name="n_users")
    .query("n_users >= @MIN_FLOW")
    .sort_values("n_users", ascending=False)
    .head(TOP_N_FLOWS)
)
print(f"Flows kept for bundling: {len(flows)}  (max volume: {flows['n_users'].max()})")

# ── BUILD NODE / EDGE TABLES FOR datashader.hammer_bundle ──────────────────
all_hexes_full = pd.unique(pd.concat([flows["home_gid9"], flows["work_gid9"]]).values)
centroids = {h: h3_to_xy(h) for h in all_hexes_full}

node_ids = {h: i for i, h in enumerate(all_hexes_full)}
nodes = pd.DataFrame({
    "id": [node_ids[h] for h in all_hexes_full],
    "x":  [centroids[h][0] for h in all_hexes_full],
    "y":  [centroids[h][1] for h in all_hexes_full],
}).set_index("id")

edges = pd.DataFrame({
    "source": flows["home_gid9"].map(node_ids).values,
    "target": flows["work_gid9"].map(node_ids).values,
    "weight": flows["n_users"].values,
})

print("Running hammer_bundle (datashader force-directed style bundling)...")
bundled = hammer_bundle(
    nodes, edges,
    decay=HB_DECAY,
    iterations=HB_ITERATIONS,
    initial_bandwidth=HB_INITIAL_BW,
    tension=HB_TENSION,
)
print("Bundling complete. Points:", len(bundled))

# bundled is a dataframe with x, y columns, paths separated by NaN rows.
# Split into individual paths.
paths = []
current = []
for _, row in bundled.iterrows():
    if pd.isna(row["x"]) or pd.isna(row["y"]):
        if current:
            paths.append(np.array(current))
            current = []
    else:
        current.append([row["x"], row["y"]])
if current:
    paths.append(np.array(current))

print(f"Number of bundled paths: {len(paths)} (should equal number of edges: {len(edges)})")

# ── DOTS — level 8, top 500 ──────────────────────────────────────────────
home_work["home_h8"] = home_work["home_gid9"].map(lambda h: h3.h3_to_parent(h, DOT_H3_LEVEL))
home_work["work_h8"] = home_work["work_gid9"].map(lambda h: h3.h3_to_parent(h, DOT_H3_LEVEL))
orig_counts = home_work.groupby("home_h8").size().reset_index(name="n_orig").rename(columns={"home_h8": "hex"})
dest_counts = home_work.groupby("work_h8").size().reset_index(name="n_dest").rename(columns={"work_h8": "hex"})
total_counts = orig_counts.merge(dest_counts, on="hex", how="outer").fillna(0)
total_counts["n_total"] = total_counts["n_orig"] + total_counts["n_dest"]
total_counts = total_counts.sort_values("n_total", ascending=False).head(TOP_N_DOTS)
dot_centroids = {h: h3_to_xy(h) for h in total_counts["hex"]}

# ── PLOT ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 11))

all_xy = np.array(list(centroids.values()))
xmin, ymin = all_xy.min(axis=0)
xmax, ymax = all_xy.max(axis=0)
pad_x = (xmax - xmin) * 0.06
pad_y = (ymax - ymin) * 0.06
ax.set_xlim(xmin - pad_x, xmax + pad_x)
ax.set_ylim(ymin - pad_y, ymax + pad_y)

try:
    ctx.add_basemap(ax, crs="EPSG:3857", source=basemaps.POSITRON,
                    zoom="auto", attribution=False)
except Exception:
    ax.set_facecolor("#f5f5f5")

n_vals = flows["n_users"].values
log_n = np.log1p(n_vals)
log_min, log_max = log_n.min(), log_n.max()

segments = []
colors = []
linewidths = []

for path, n, ln in zip(paths, n_vals, log_n):
    t_val = (ln - log_min) / (log_max - log_min + 1e-9)
    lw = LINE_WIDTH_MIN + t_val * (LINE_WIDTH_MAX - LINE_WIDTH_MIN)
    alpha = BASE_ALPHA * (0.3 + 0.7 * t_val)

    n_pts = len(path)
    if n_pts < 2:
        continue
    for k in range(n_pts - 1):
        tt = k / (n_pts - 2) if n_pts > 2 else 0
        c = (1 - tt) * ORIGIN_COLOR + tt * DEST_COLOR
        segments.append([path[k], path[k + 1]])
        colors.append(np.append(c, alpha))
        linewidths.append(lw)

lc = LineCollection(segments, colors=colors, linewidths=linewidths,
                     capstyle="round", joinstyle="round", zorder=2)
ax.add_collection(lc)

# Dots
sizes = np.sqrt(total_counts["n_total"].values)
sizes_scaled = np.clip(sizes / sizes.max() * 200, 8, 200)
xy_dots = np.array([dot_centroids[h] for h in total_counts["hex"]])
ax.scatter(xy_dots[:, 0], xy_dots[:, 1], s=sizes_scaled, c=DOT_COLOR,
           alpha=0.85, linewidths=0.4, edgecolors="white", zorder=4)

legend_elements = [
    Line2D([0], [0], color=ORIGIN_COLOR, lw=4, label="Origin (home)"),
    Line2D([0], [0], color=DEST_COLOR, lw=4, label="Destination (work)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=DOT_COLOR,
           markersize=10, label="Hexagon (size = total trips)"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=10, frameon=False)

ax.set_title("Helsinki — home-to-work commuting flows (edge-bundled)",
             fontsize=13, pad=10)
ax.set_axis_off()

plt.tight_layout()
plt.savefig("./output/fig2_od_flows_helsinki_hammer.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: ./output/fig2_od_flows_helsinki_hammer.png")

In [ ]:
pip install git+https://github.com/zorzalerrante/aves.git


In [ ]:
"""
Figure 2 — Helsinki home-to-work OD flow map using AVES
(Network + NodeLink + force-directed edge bundling + GeoFacetGrid basemap)

Data: users_and_stays_3months.parquet (home_gid9, work_gid9, user_id)
Boundary: helsinki_filter_map.geojson — used ONLY to filter, never drawn.
"""

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from h3 import h3
from pyproj import Transformer
from shapely.geometry import Point

from aves.models.network import Network
from aves.visualization.networks import NodeLink
from aves.visualization.figures import GeoFacetGrid

TR = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

# ── CONFIG ──────────────────────────────────────────────────────────────────
USERS_PATH      = "scratch/data/users_and_stays_3months.parquet"
FILTER_GEOJSON  = "./data/helsinki_filter_map.geojson"

MIN_FLOW    = 1
TOP_N_FLOWS = 1000

def h3_to_xy(h3_id):
    lat, lon = h3.h3_to_geo(h3_id)
    return TR.transform(lon, lat)

# ── LOAD DATA ───────────────────────────────────────────────────────────────
print("Loading Helsinki users...")
users = pd.read_parquet(USERS_PATH)
users = users[users["home_gid9"].notna() & users["work_gid9"].notna()]

home_work = (
    users[["user_id", "home_gid9", "work_gid9"]]
    .drop_duplicates(subset="user_id")
    .copy()
)
home_work = home_work[home_work["home_gid9"] != home_work["work_gid9"]]

# ── SPATIAL FILTER: keep only pairs where BOTH home and work are inside
#    the boundary geojson. Boundary is used purely as a mask, never plotted.
boundary_gdf = gpd.read_file(FILTER_GEOJSON).to_crs("EPSG:4326")
boundary = boundary_gdf.union_all()

unique_hexes = pd.unique(pd.concat([home_work["home_gid9"], home_work["work_gid9"]]).values)
inside_boundary = {}
for h in unique_hexes:
    lat, lon = h3.h3_to_geo(h)
    inside_boundary[h] = boundary.contains(Point(lon, lat))

home_work["home_inside"] = home_work["home_gid9"].map(inside_boundary)
home_work["work_inside"] = home_work["work_gid9"].map(inside_boundary)
home_work = home_work[home_work["home_inside"] & home_work["work_inside"]].drop(
    columns=["home_inside", "work_inside"]
)
print(f"Home-work pairs after boundary filter: {len(home_work)}")

# ── AGGREGATE FLOWS (edge list for AVES Network) ────────────────────────────
flows = (
    home_work
    .groupby(["home_gid9", "work_gid9"])
    .size()
    .reset_index(name="n_users")
    .query("n_users >= @MIN_FLOW")
    .sort_values("n_users", ascending=False)
    .head(TOP_N_FLOWS)
)
print(f"Flows kept: {len(flows)}  (max volume: {flows['n_users'].max()})")

# ── BUILD NODE POSITIONS (projected, EPSG:3857) ─────────────────────────────
all_hexes = pd.unique(pd.concat([flows["home_gid9"], flows["work_gid9"]]).values)
node_xy = {h: h3_to_xy(h) for h in all_hexes}

node_positions = pd.DataFrame({
    "node_id": list(node_xy.keys()),
    "x": [v[0] for v in node_xy.values()],
    "y": [v[1] for v in node_xy.values()],
}).set_index("node_id")

node_positions_gdf = gpd.GeoDataFrame(
    node_positions,
    geometry=gpd.points_from_xy(node_positions["x"], node_positions["y"]),
    crs="EPSG:3857",
)

# ── BUILD AVES NETWORK FROM EDGELIST ────────────────────────────────────────
zone_od_network = Network.from_edgelist(
    flows, source="home_gid9", target="work_gid9", weight="n_users"
)

zone_nodelink = NodeLink(zone_od_network)
zone_nodelink.layout_nodes(method="geographical", geodataframe=node_positions_gdf)
zone_nodelink.bundle_edges(
    method="force-directed", K=1, S=500, I=30, compatibility_threshold=0.65, C=6
)
zone_nodelink.set_node_drawing(
    "plain", weights=zone_od_network.node_degree("total")
)
zone_nodelink.set_edge_drawing(method="origin-destination")


def plot_network(ax, geo_data, *args, **kwargs):
    zone_nodelink.plot(ax, *args, **kwargs)


# ── PLOT WITH GeoFacetGrid ───────────────────────────────────────────────────
grid = GeoFacetGrid(boundary_gdf, context=boundary_gdf, height=10)
grid.add_layer(boundary_gdf, facecolor="#f7f7f7", edgecolor="none")
grid.add_layer(
    plot_network,
    nodes=dict(color="#2b2b2b", edgecolor="white", node_size=80, alpha=0.85),
    edges=dict(linewidth=0.6, alpha=0.35),
)
grid.set_title("Helsinki — home-to-work commuting flows (AVES, force-directed bundling)")

plt.savefig("./output/fig2_od_flows_helsinki_aves.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: ./output/fig2_od_flows_helsinki_aves.png")

In [ ]:
conda install -c conda-forge graph-tool